In [ ]:

import torch
import torch.nn as nn
from transformers import BertModel
from transformers import AutoModelForMaskedLM



class RegressionBert(nn.Module):
    def __init__(self):
        super(RegressionBert, self).__init__()
        # Load the plain BERT engine --> this is GERMAN bert
        self.bert = AutoModelForMaskedLM.from_pretrained("deepset/gbert-base", dtype="auto")
        # Add your own custom layers




        self.drop = nn.Dropout(p=0.1)
        self.out = nn.Linear(self.bert.config.hidden_size, 1)
    
    def forward(self, input_ids, attention_mask):
        # BERT returns a dictionary or tuple
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # 'pooler_output' is the [CLS] token representation (shape: [batch_size, 768])
        pooled_output = outputs.pooler_output
        
        # Pass through our custom layers
        output = self.drop(pooled_output)
        return self.out(output)
    
    def unfreeze_bert_layers(self, num_layers): 
        
        for i in range(12-num_layers, 12): 
            for param in self.bert.encoder.layer[i].parameters(): 
                param.requires_grad = True

# Initialize for 6 emotion classes
model = RegressionBert()            



Some weights of the model checkpoint at deepset/gbert-base were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:

def train(
    model, 
    train_loader, 
    epochs, 
    learning_rate, 
    device
):
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.3, total_iters=10)

    loss_fn = nn.MSELoss().to(device)


    for param in model.parameters():
        param.requires_grad = False # freeze everything at start. 

    for param in model.out.parameters():  
        param.requires_grad = True #: unfreeze the top layer at the start. 


    
    for param in model.bert.pooler.parameters():  
        param.requires_grad = True #: unfreeze the pooler from BERT (this is used for classification/ regression)


    for epoch in range(epochs): 
        running_loss = 0
        for step, batch in enumerate(train_loader): 
            inputs, mask, labels = batch
            inputs, mask,  labels = inputs.to(device), mask.to(device), labels.to(device) 

            optimizer.zero_grad()
            

            out = model(input_ids=inputs, attention_mask=mask)

            freq = 2 # controls how many epochs are needed to unfreeze a layer. 
            num_layers = min(int(epoch/freq), 12)

            model.unfreeze_bert_layers()

            loss = loss_fn(out, labels)
            runnig_loss+= loss.item()
            loss.backward()
            #: to avoid exploding gradients. 
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            
            optimizer.step()
        #: recordar que el scheduler step se hace POR CADA EPOCH
        scheduler.step()



    loss.backward()
    optimizer.step()

In [37]:
print(list(model.bert.named_children()))
#print(list(encoder.named_children()))

[('bert', BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31102, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inp

In [4]:

from transformers import AutoModel , AutoTokenizer, AutoModelForMaskedLM


model = AutoModelForMaskedLM.from_pretrained("distilbert/distilbert-base-german-cased")

print(list(model.named_children()))

[('activation', GELUActivation()), ('distilbert', DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(31102, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768,